In [11]:
import polars as pl

# === Đọc file Parquet ===
path = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\dataset\sales_pers.item_chunk_0.parquet"
df = pl.read_parquet(path)
import polars as pl

# === Kiểm tra sự tồn tại của hai cột ===
for col in ["description", "description_new"]:
    if col not in df.columns:
        raise ValueError(f"❌ Cột '{col}' không tồn tại trong dataframe!")

# === Regex pattern tìm các biến thể của "step 1" hoặc "bước 1" ===
# Giải thích:
#   - step\s*[-_]?\s*1 → tìm "step 1", "step-1", "step_1", "step1"
#   - bước\s*1 hoặc bước\s*1 → tìm cả "bước 1" và "bước 1" (có dấu hoặc không)
pattern = r"(step\s*[-_]?\s*1|bước\s*1|bước\s*1)"

# === Lọc các dòng có chứa pattern trong 1 trong 2 cột ===
df_check = df.filter(
    pl.col("description").cast(pl.Utf8).str.to_lowercase().str.contains(pattern)
    | pl.col("description_new").cast(pl.Utf8).str.to_lowercase().str.contains(pattern)
)

# === Thống kê số lượng ===
count_step1 = df_check.height
total_rows = df.height
pct = round(100 * count_step1 / total_rows, 2)

print(f"✅ Có {count_step1:,} dòng (≈ {pct}%) chứa 'step 1' hoặc 'bước 1' (mọi cách viết) trong mô tả.")


✅ Có 230 dòng (≈ 0.84%) chứa 'step 1' hoặc 'bước 1' (mọi cách viết) trong mô tả.


In [12]:
pl.Config.set_tbl_rows(20) 
pl.Config.set_tbl_cols(20) 
pl.Config.set_tbl_width_chars(150)
pl.Config.set_fmt_str_lengths(1000)

polars.config.Config

In [14]:

# === In 5 dòng ví dụ ===
print("\n=== Ví dụ 5 dòng có chứa 'step 1' hoặc 'bước 1' ===")
for row in df_check.select(["item_id", "category", "description", "description_new"]).head(10).to_dicts():
    print(f"- item_id: {row.get('item_id')}")
    print(f"  category: {row.get('category')}")
    desc = (row.get('description') or "")[:120].replace("\n", " ")
    desc_merge = (row.get('description_new') or "")[:120].replace("\n", " ")
    print(f"  description: {desc}...")
    print(f"  description_new: {desc_merge}...\n")



=== Ví dụ 5 dòng có chứa 'step 1' hoặc 'bước 1' ===
- item_id: 3806000000012
  category: Bàn chải điện
  description: ﻿Bàn chải chữ Animo phù hợp sử dụng cho bé từ 2 tuổi. Sản phẩm được làm từ chất liệu mềm mại, an toàn và có thiết kế đầu...
  description_new: Chi tiết sản phẩmTên sản phẩm:Bàn chải điện chữ U Animo (Xanh, GH-TR858)Thương hiệu: AnimoSản xuất tại: Trung QuốcĐộ tuổ...

- item_id: 5051000000003
  category: Gối nằm cao su Animo
  description: ﻿﻿﻿Gối memory foam (cao su non) lớn Animo B2305_DQ002 mang lại cảm giác mềm mại và êm ái cho bé khi ngủ. Gối được làm từ...
  description_new: Chi tiết sản phẩm                     Tên sản phẩm: Gối memory foam (cao su non) lớn Animo B2305_DQ002 (PUB122,Xanh)    ...

- item_id: 0006020000207
  category: Sữa tắm Cung đình
  description: THÔNG TIN SẢN PHẨM Tên sản phẩm: Đai quấn muối Cung Đình Thương hiệu: Làm đẹp Cung Đình Xuất xứ: Việt Nam HƯỚNG DẪN SỬ D...
  description_new: Chi tiết sản phẩm                     Tên sản phẩm: Đai quấ

In [15]:
import polars as pl

# === Đọc file Parquet ===
path = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\dataset\sales_pers.item_chunk_0.parquet"
df = pl.read_parquet(path)

# === Kiểm tra sự tồn tại của 2 cột ===
for col in ["description", "description_new"]:
    if col not in df.columns:
        raise ValueError(f"❌ Cột '{col}' không tồn tại trong dataframe!")

# === Regex pattern tìm các biến thể của "mom", "mẹ", "me bau", "bà bầu" ===
# Giải thích:
#   - mom → cho tiếng Anh
#   - mẹ|me\b → tìm “mẹ” hoặc “me” (có dấu hoặc không)
#   - me bau|mẹ bầu|bà bầu → tìm các cụm dành cho mẹ bầu
#   - mam|mama → có thể xuất hiện trong vài brand (ví dụ: mama milk)
pattern = r"(mom|mẹ|me\b|me\s*bầu|mẹ\s*bầu|bà\s*bầu|mam|mama)"

# === Lọc các dòng có chứa pattern trong 1 trong 2 cột mô tả ===
df_mom = df.filter(
    pl.col("description").cast(pl.Utf8).str.to_lowercase().str.contains(pattern)
    | pl.col("description_new").cast(pl.Utf8).str.to_lowercase().str.contains(pattern)
)

# === Thống kê số lượng ===
count_mom = df_mom.height
total_rows = df.height
pct = round(100 * count_mom / total_rows, 2)

print(f"✅ Có {count_mom:,} dòng (≈ {pct}%) chứa từ khóa 'mom', 'mẹ', 'mẹ bầu', hoặc tương tự trong mô tả.")

# === In 10 dòng ví dụ ===
print("\n=== Ví dụ 10 dòng có chứa từ khóa mẹ/mom/bầu ===")
for row in df_mom.select(["item_id", "category", "description", "description_new"]).head(10).to_dicts():
    print(f"- item_id: {row.get('item_id')}")
    print(f"  category: {row.get('category')}")
    desc = (row.get('description') or "")[:120].replace("\n", " ")
    desc_new = (row.get('description_new') or "")[:120].replace("\n", " ")
    print(f"  description: {desc}...")
    print(f"  description_new: {desc_new}...\n")


✅ Có 5,952 dòng (≈ 21.78%) chứa từ khóa 'mom', 'mẹ', 'mẹ bầu', hoặc tương tự trong mô tả.

=== Ví dụ 10 dòng có chứa từ khóa mẹ/mom/bầu ===
- item_id: 0020010000094
  category: Merries_Sơ Sinh
  description: ﻿﻿Tã dán Merries size S 82 miếng là sản phẩm dành cho bé 4-8kg đến từ thương hiệu uy tín Merries của Nhật Bản. Ra đời vớ...
  description_new: Không xác định...

- item_id: 0024181040235
  category: Áo bé trai
  description: Áo thun bé trai tay ngắn CF B078010 Xanh  là sản phẩm với chất liệu cotton mềm mại, thấm hút mồ hôi, đường may kỹ lưỡng ...
  description_new: Không xác định...

- item_id: 0008180000017
  category: Thảm xốp mảnh ghép
  description: - Sản phẩm dành cho bé từ 3 - 36 tháng tuổi. - Nguyên liệu được làm bằng cao su có tính đàn hồi tốt, dùng lót sàn. Giúp ...
  description_new: Chi tiết sản phẩm                     Tên sản phẩm: Thảm xốp hình chữ 30x30cm, 26 miếng         Chất liệu: Cao su       ...

- item_id: 0020010000151
  category: Moony_Sơ Sinh
  description: 

In [16]:
import polars as pl

# === Đọc file Parquet ===
path = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\dataset\sales_pers.item_chunk_0.parquet"
df = pl.read_parquet(path)

# === Kiểm tra sự tồn tại của 2 cột ===
for col in ["description", "description_new"]:
    if col not in df.columns:
        raise ValueError(f"❌ Cột '{col}' không tồn tại trong dataframe!")

# === Regex tìm các sản phẩm sữa cho mẹ/mom/bầu ===
# Phần 1: milk_pattern → tìm các sản phẩm là sữa
# Phần 2: mom_pattern  → tìm từ khóa liên quan đến mẹ, mom, bầu, v.v.
milk_pattern = r"(sữa|milk|formula|dairy)"
mom_pattern = r"(mom|mẹ|me\b|mẹ\s*bầu|me\s*bầu|bà\s*bầu|mam|mama|maternity|pregnan)"

# === Kết hợp điều kiện ===
# Chỉ giữ các dòng chứa cả hai nhóm từ khóa (milk + mom)
df_milk_mom = df.filter(
    (
        pl.col("description").cast(pl.Utf8).str.to_lowercase().str.contains(milk_pattern)
        | pl.col("description_new").cast(pl.Utf8).str.to_lowercase().str.contains(milk_pattern)
    )
    &
    (
        pl.col("description").cast(pl.Utf8).str.to_lowercase().str.contains(mom_pattern)
        | pl.col("description_new").cast(pl.Utf8).str.to_lowercase().str.contains(mom_pattern)
    )
)

# === Thống kê ===
count_milk_mom = df_milk_mom.height
total_rows = df.height
pct = round(100 * count_milk_mom / total_rows, 2)

print(f"✅ Có {count_milk_mom:,} dòng (≈ {pct}%) là SỮA cho mẹ/mom/bầu (milk + mom keywords).")

# === In ví dụ 10 dòng ===
print("\n=== Ví dụ 10 dòng sữa cho mẹ/bầu ===")
for row in df_milk_mom.select(["item_id", "category", "brand", "description", "description_new"]).head(10).to_dicts():
    print(f"- item_id: {row.get('item_id')}")
    print(f"  brand: {row.get('brand')}")
    print(f"  category: {row.get('category')}")
    desc = (row.get('description') or "")[:120].replace("\n", " ")
    desc_new = (row.get('description_new') or "")[:120].replace("\n", " ")
    print(f"  description: {desc}...")
    print(f"  description_new: {desc_new}...\n")


✅ Có 1,368 dòng (≈ 5.01%) là SỮA cho mẹ/mom/bầu (milk + mom keywords).

=== Ví dụ 10 dòng sữa cho mẹ/bầu ===
- item_id: 0020010000438
  brand: Meiji Nhập khẩu
  category: Meiji Step 1
  description: ﻿﻿﻿﻿﻿﻿Sữa Meiji Infant Formula 800g (0-12 tháng) là sữa bột công thức được nhập khẩu chính hãng từ Nhật Bản. Sản phẩm dà...
  description_new: Không xác định...

- item_id: 0020010000440
  brand: Meiji Nhập khẩu
  category: Meiji Step 3
  description: ﻿Sữa Meiji Growing up Formula 800g (12-36 tháng) thuộc thương hiệu Meiji nổi tiếng hàng đầu Nhật Bản, cung cấp các vitam...
  description_new: Không xác định...

- item_id: 0020010000492
  brand: Wakodo
  category: Wakodo Mom
  description: ﻿WAKODO MOM là dòng sản phẩm cao cấp dành cho mẹ mang thai và cho con bú, giúp xây dựng nền tảng bền vững cho con ngay t...
  description_new: Không xác định...

- item_id: 0007080000415
  brand: Imedicare
  category: Nhiệt kế ngừng bán
  description: Nhiệt kế hồng ngoại đa chức năng iMediCare iTM 8F là thi